<div style="text-align: center; padding: 30px; background: linear-gradient(135deg, #1e1b4b 0%, #4338ca 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);">
  <h1 style="color: white; margin: 0 0 8px 0; font-size: 2.5em;">🎙️ Watch The Book — Unified TTS & Audit</h1>
  <h3 style="color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;">Advanced Audio Production Server</h3>
  <h5 style="color: #ddd; margin: 0 0 20px 0;">Qwen3-TTS 1.7B | Stable-TS High-Precision Aligner | GPU/CPU Optimized</h5>
  <div style="display: flex; justify-content: center; gap: 15px;">
    <a href="https://www.youtube.com/@WatchTheBook?sub_confirmation=1" target="_blank" style="background: #ef4444; color: white; padding: 10px 20px; border-radius: 8px; text-decoration: none; font-weight: bold;">▶ YouTube</a>
  </div>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/Colab-Free%20Tier-orange?style=for-the-badge&logo=googlecolab&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4_GPU_Enabled-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
</div>

In [ ]:
# @title Step 0: 🔑 Setup Hugging Face Credentials & Variables
#@markdown ### Enter your private repository settings below.
#@markdown ---
HF_TOKEN = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx" #@param {type:"string"}
HF_REPO_ID = "user/repo" #@param {type:"string"}

print("✅ Credentials configured successfully!")
print(f"🔗 Target Repository: {HF_REPO_ID}")

In [ ]:
# @title Step 1: 🔍 Hardware Verification
import torch
import os

# Suppress technical chatter for cleaner logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ADJUST_HUGE_PAGES"] = "0"

IS_GPU = torch.cuda.is_available()

if IS_GPU:
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\n✅ GPU detected: {gpu_name}")
    print(f"✅ VRAM: {vram_total:.1f} GB")
else:
    print("ℹ️ No GPU detected. Notebook will run on CPU mode (Standard Speed).")

In [ ]:
# @title Step 2: 📦 Install Dependencies
print("📦 Synchronizing Production Environment...")

# 1. Install Qwen-TTS and core transformers first to establish stable versioning
!pip install -q qwen-tts accelerate transformers --upgrade

# 2. Install Stable-TS from PyPI, Faster-Whisper, and PEFT for LoRA loading
!pip install -q stable-ts faster-whisper peft

# 3. Audio and UI processing tools
!pip install -q "gradio>=5.0.0" soundfile

print("\n✅ Environment Ready!")

In [ ]:
# @title Step 3: 🚀 Launch Unified Gradio Server

import os
import shutil
import gradio as gr
import torch
import soundfile as sf
import tempfile
import gc
import time
import json
import numpy as np
import stable_whisper
import transformers.utils.hub
import huggingface_hub
from peft import PeftModel

# ── Global Production State & Arbiter ──
current_tts_model = None
current_tts_type = None
current_tts_subfolder = None  # Tracks active custom speaker checkpoint folder
active_loading_subfolder = None  # Track subfolder dynamically chosen in Gradio/manifest
current_tts_model_path = None  # Holds the absolute disk path of the active model
current_audit_model = None

# ── Monkey-Patching Upstream qwen_tts Bug ──
_orig_cached_file = transformers.utils.hub.cached_file

def _patched_cached_file(repo_id, filename, **kwargs):
    if repo_id == HF_REPO_ID and filename.startswith("speech_tokenizer/"):
        print(f"🔧 [Patch] Redirecting speech_tokenizer file '{filename}' to official Qwen Base...", flush=True)
        kwargs.pop("subfolder", None)
        return _orig_cached_file("Qwen/Qwen3-TTS-12Hz-1.7B-Base", filename, **kwargs)
        
    if repo_id == HF_REPO_ID and not kwargs.get("subfolder"):
        if active_loading_subfolder:
            print(f"🔧 [Patch] Restoring popped subfolder: '{active_loading_subfolder}' for file: '{filename}'", flush=True)
            kwargs["subfolder"] = active_loading_subfolder
    return _orig_cached_file(repo_id, filename, **kwargs)

transformers.utils.hub.cached_file = _patched_cached_file
huggingface_hub.cached_file = _patched_cached_file

try:
    import qwen_tts.core.models.modeling_qwen3_tts
    qwen_tts.core.models.modeling_qwen3_tts.cached_file = _patched_cached_file
except Exception:
    pass

# ── Monkey-Patching PeftModel Config Delegation ──
@property
def _patched_peft_config(self):
    try:
        return self.get_base_model().config
    except Exception:
        if hasattr(self, "base_model") and hasattr(self.base_model, "model"):
            return self.base_model.model.config
    return None

PeftModel.config = _patched_peft_config

# ── Dynamic Hugging Face Class Registration ──
from transformers import AutoConfig, AutoModel, AutoProcessor, AutoTokenizer, AutoFeatureExtractor
from qwen_tts.core.models import Qwen3TTSConfig, Qwen3TTSForConditionalGeneration, Qwen3TTSProcessor
from qwen_tts.core.models.configuration_qwen3_tts import Qwen3TTSTalkerConfig
from qwen_tts import Qwen3TTSModel

AutoConfig.register("qwen3_tts", Qwen3TTSConfig)
AutoModel.register(Qwen3TTSConfig, Qwen3TTSForConditionalGeneration)
AutoProcessor.register(Qwen3TTSConfig, Qwen3TTSProcessor)

# ── Redirect Auto-Processors to the pristine Official Qwen Base ──
_orig_autoprocessor_from_pretrained = AutoProcessor.from_pretrained
_orig_autotokenizer_from_pretrained = AutoTokenizer.from_pretrained
_orig_autofeatureextractor_from_pretrained = AutoFeatureExtractor.from_pretrained

def _patched_autoprocessor_from_pretrained(pretrained_model_name_or_path, *args, **kwargs):
    if pretrained_model_name_or_path == HF_REPO_ID:
        print("🔧 [Patch] Redirecting AutoProcessor to official Qwen Base...", flush=True)
        kwargs.pop("subfolder", None)
        return _orig_autoprocessor_from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", *args, **kwargs)
    return _orig_autoprocessor_from_pretrained(pretrained_model_name_or_path, *args, **kwargs)

def _patched_autotokenizer_from_pretrained(pretrained_model_name_or_path, *args, **kwargs):
    if pretrained_model_name_or_path == HF_REPO_ID:
        print("🔧 [Patch] Redirecting AutoTokenizer to official Qwen Base...", flush=True)
        kwargs.pop("subfolder", None)
        return _orig_autotokenizer_from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", *args, **kwargs)
    return _orig_autotokenizer_from_pretrained(pretrained_model_name_or_path, *args, **kwargs)

def _patched_autofeatureextractor_from_pretrained(pretrained_model_name_or_path, *args, **kwargs):
    if pretrained_model_name_or_path == HF_REPO_ID:
        print("🔧 [Patch] Redirecting AutoFeatureExtractor to official Qwen Base...", flush=True)
        kwargs.pop("subfolder", None)
        return _orig_autofeatureextractor_from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", *args, **kwargs)
    return _orig_autofeatureextractor_from_pretrained(pretrained_model_name_or_path, *args, **kwargs)

AutoProcessor.from_pretrained = _patched_autoprocessor_from_pretrained
AutoTokenizer.from_pretrained = _patched_autotokenizer_from_pretrained
AutoFeatureExtractor.from_pretrained = _patched_autofeatureextractor_from_pretrained

from typing import List, Dict, Tuple, Optional, Any
from huggingface_hub import login, snapshot_download, hf_hub_download

# ── Hugging Face Private Repository Authentication ──
login(token=HF_TOKEN)

# ── Hardware & Compute Logic ──
IS_GPU = torch.cuda.is_available()
DEVICE = "cuda:0" if IS_GPU else "cpu"
DTYPE_TTS = torch.bfloat16 if IS_GPU else torch.float32
COMPUTE_WHISPER = "float16" if IS_GPU else "int8"
HW_STATUS = "GPU High-Speed ⚡" if IS_GPU else "CPU Standard 🐌"

if IS_GPU:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

# VRAM Guard: Keep both models resident in memory if GPU has at least 10 GB of total VRAM
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3) if IS_GPU else 0.0
KEEP_WARM = IS_GPU and VRAM_GB >= 10.0

MODEL_MAP = {
    "base": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
    "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    "finetuned": HF_REPO_ID,  # Target repository resolved dynamically from form settings
}

def clear_gpu_cache():
    gc.collect()
    if IS_GPU:
        torch.cuda.empty_cache()

def unload_audit_model():
    global current_audit_model
    if current_audit_model is not None:
        print("♻️ Unloading Aligner to free VRAM...", flush=True)
        del current_audit_model
        current_audit_model = None
        clear_gpu_cache()

def unload_tts_model():
    global current_tts_model, current_tts_type, current_tts_subfolder
    if current_tts_model is not None:
        subfolder_desc = f" [{current_tts_subfolder}]" if current_tts_subfolder else ""
        print(f"♻️ Unloading TTS ({current_tts_type}{subfolder_desc}) model...", flush=True)
        del current_tts_model
        current_tts_model = None
        current_tts_type = None
        current_tts_subfolder = None
        clear_gpu_cache()

def load_tts_model(model_type, subfolder=None):
    global current_tts_model, current_tts_type, current_tts_subfolder, active_loading_subfolder, current_tts_model_path
    if current_tts_type == model_type and current_tts_subfolder == subfolder: 
        return current_tts_model

    def _load_raw_base(b_type):
        b_name = MODEL_MAP[b_type]
        b_kwargs = {
            "device_map": DEVICE,
            "dtype": DTYPE_TTS,
            "attn_implementation": "sdpa",
        }
        if not IS_GPU:
            b_kwargs["device_map"] = "cpu"
        return Qwen3TTSModel.from_pretrained(b_name, **b_kwargs)

    target_model_type = model_type
    target_subfolder = subfolder
    
    unload_tts_model()
    if not KEEP_WARM:
        unload_audit_model()
    
    model_name = MODEL_MAP[model_type]
    active_loading_subfolder = subfolder
    
    try:
        kwargs = {
            "device_map": DEVICE,
            "dtype": DTYPE_TTS,
            "attn_implementation": "sdpa",
        }
        
        print("📥 Pre-downloading and patching official Qwen Base...", flush=True)
        base_dir = snapshot_download(
            repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
            allow_patterns=[
                "speech_tokenizer/*",
                "preprocessor_config.json",
                "config.json",
                "tokenizer_config.json",
                "vocab.json",
                "merges.txt"
            ]
        )
        
        base_tokenizer_dir = os.path.join(base_dir, "speech_tokenizer")
        os.makedirs(base_tokenizer_dir, exist_ok=True)
        for config_file in ["preprocessor_config.json", "config.json"]:
            src = os.path.join(base_dir, config_file)
            dst = os.path.join(base_tokenizer_dir, config_file)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)
        
        if subfolder:
            escaped_subfolder = subfolder.replace("[", "[[]").replace("]", "[]]")
            
            from huggingface_hub import HfApi
            api = HfApi(token=HF_TOKEN)
            refs = api.list_repo_refs(repo_id=model_name)
            subfolder_files = [b.name for b in refs.branches if b.name != "main"]
            
            is_lora = "alexandria" in subfolder
            
            if is_lora:
                print(f"📥 Pre-downloading finetuned LoRA branch: {model_name} -> revision '{subfolder}'...", flush=True)
                repo_dir = snapshot_download(
                    repo_id=model_name,
                    revision=subfolder,  # Target the branch name as the revision
                    allow_patterns=[
                        "adapter_config.json",
                        "adapter_model.safetensors",
                        "adapter_model.bin",
                        "config.json",
                        "speaker_embedding.safetensors",
                        "ref_sample.wav",
                        "ref.wav"
                    ],
                    token=HF_TOKEN
                )
                local_model_path = repo_dir
                
                with open(os.path.join(local_model_path, "adapter_config.json"), "r") as f:
                    peft_config = json.load(f)
                
                auto_mapping_obj = peft_config.get("auto_mapping")
                base_class = ""
                if isinstance(auto_mapping_obj, dict):
                    base_class = auto_mapping_obj.get("base_model_class", "")
                is_talker_lora = "Talker" in base_class or peft_config.get("task_type") is None
                
                base_model_path = peft_config.get("base_model_name_or_path")
                if not base_model_path:
                    base_model_path = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
                
                base_type = "base"
                if "CustomVoice" in base_model_path:
                    base_type = "custom"
                elif "VoiceDesign" in base_model_path:
                    base_type = "design"
                    
                print(f"📥 Loading base model for LoRA SFT: '{base_model_path}'...", flush=True)
                base_model = _load_raw_base(base_type)
                
                from peft import PeftModel
                print(f"📁 [PEFT] Injecting LoRA adapter weights onto the inner Talker module: '{local_model_path}'", flush=True)
                base_model.model.talker = PeftModel.from_pretrained(
                    base_model.model.talker,
                    local_model_path,
                    token=HF_TOKEN
                )
                
                if isinstance(base_model.model, PeftModel):
                    core_model = base_model.model.get_base_model()
                else:
                    core_model = base_model.model
                
                config_path = os.path.join(local_model_path, "config.json")
                if os.path.exists(config_path):
                    with open(config_path, "r") as f:
                        custom_config = json.load(f)
                else:
                    custom_config = {}
                
                for k, v in custom_config.items():
                    if k != "talker_config":
                        setattr(core_model.config, k, v)
                
                if getattr(core_model.config, "talker_config", None) is None:
                    core_model.config.talker_config = Qwen3TTSTalkerConfig()
                
                talker_cfg_obj = core_model.config.talker_config
                custom_talker_dict = custom_config.get("talker_config", {})
                if isinstance(custom_talker_dict, dict):
                    if "spk_id" in custom_talker_dict:
                        talker_cfg_obj.spk_id = custom_talker_dict["spk_id"]
                    if "spk_is_dialect" in custom_talker_dict:
                        talker_cfg_obj.spk_is_dialect = custom_talker_dict["spk_is_dialect"]

                if not custom_config:
                    spk_id_key = subfolder
                    spk_id_val = 3000
                    if talker_cfg_obj.spk_id is None:
                        talker_cfg_obj.spk_id = {}
                    talker_cfg_obj.spk_id[spk_id_key] = spk_id_val
                    
                    if talker_cfg_obj.spk_is_dialect is None:
                        talker_cfg_obj.spk_is_dialect = {}
                    talker_cfg_obj.spk_is_dialect[spk_id_key] = False

                spk_keys = list(talker_cfg_obj.spk_id.keys())
                core_model.supported_speakers = spk_keys
                if hasattr(base_model.model, "supported_speakers"):
                    base_model.model.supported_speakers = spk_keys
                
                if hasattr(base_model.model, "base_model") and hasattr(base_model.model.base_model, "model"):
                    base_model.model.base_model.model.tts_model_type = "custom_voice"
                if hasattr(core_model, "tts_model_type"):
                    core_model.tts_model_type = "custom_voice"

                spk_embedding = None
                emb_path = os.path.join(local_model_path, "speaker_embedding.safetensors")
                
                if os.path.exists(emb_path):
                    try:
                        from safetensors.torch import load_file
                        emb_dict = load_file(emb_path)
                        if "speaker_embedding" in emb_dict:
                            spk_embedding = emb_dict["speaker_embedding"]
                            print("📁 [Emb] Loaded pre-computed speaker embedding from safetensors.", flush=True)
                    except Exception as e:
                        print(f"⚠️ [Emb Warning] Failed to load safetensors embedding: {e}", flush=True)

                if spk_embedding is None:
                    ref_names_list = ["ref_sample.wav", "ref.wav"]
                    ref_path = None
                    for r_name in ref_names_list:
                        cand = os.path.join(local_model_path, r_name)
                        if os.path.exists(cand):
                            ref_path = cand
                            break
                    if ref_path:
                        try:
                            print(f"🎤 [Emb] Extracting speaker embedding dynamically from '{ref_path}'...", flush=True)
                            with torch.inference_mode():
                                audio_data, sr_data = sf.read(ref_path)
                                if len(audio_data.shape) > 1:
                                    audio_data = np.mean(audio_data, axis=1)
                                audio_tensor = torch.from_numpy(audio_data).float().to(DEVICE).unsqueeze(0)
                                
                                if hasattr(base_model, "model") and hasattr(base_model.model, "speaker_encoder"):
                                    spk_embedding = base_model.model.speaker_encoder(audio_tensor)
                                elif hasattr(base_model, "speaker_encoder"):
                                    spk_embedding = base_model.speaker_encoder(audio_tensor)
                                else:
                                    print("❌ [Emb Error] Could not locate 'speaker_encoder' module on the model.", flush=True)
                        except Exception as e:
                            print(f"❌ [Emb Error] Dynamic extraction failed: {e}", flush=True)

                if spk_embedding is not None:
                    try:
                        spk_embedding = spk_embedding.to(device=DEVICE, dtype=DTYPE_TTS)
                        spk_id_key = list(talker_cfg_obj.spk_id.keys())[0] if talker_cfg_obj.spk_id else "stephen_fry_lora_en"
                        spk_id_val = talker_cfg_obj.spk_id[spk_id_key] if talker_cfg_obj.spk_id else 3000
                        
                        injected = False
                        for name, module in core_model.named_modules():
                            if "speaker" in name.lower() and hasattr(module, "weight") and isinstance(module.weight, torch.nn.Parameter):
                                if module.weight.shape[-1] == 2048:
                                    try:
                                        with torch.no_grad():
                                            num_embeddings = module.weight.shape[0]
                                            if spk_id_val < num_embeddings:
                                                module.weight[spk_id_val].copy_(spk_embedding)
                                                print(f"✅ [Emb Injection] Successfully injected tensor into module: {name} at index {spk_id_val}", flush=True)
                                                injected = True
                                            else:
                                                print(f"⚠️ [Emb Injection Warning] index {spk_id_val} out of bounds for module: {name} (shape: {module.weight.shape})", flush=True)
                                    except Exception as copy_err:
                                        pass
                        if not injected:
                            print("⚠️ [Emb Injection] No compatible speaker embedding layer found in target modules.", flush=True)
                    except Exception as e:
                        print(f"❌ [Emb Injection Error] Failed to inject embedding: {e}", flush=True)

                current_tts_model = base_model
            else:
                print(f"📥 Pre-downloading finetuned Full SFT branch: {model_name} -> revision '{subfolder}'...", flush=True)
                repo_dir = snapshot_download(
                    repo_id=model_name,
                    revision=subfolder,  # Target the branch name as the revision
                    allow_patterns=[
                        "config.json",
                        "generation_config.json",
                        "model.safetensors",
                        "ref_sample.wav",
                        "ref.wav",
                        "ref_sample.txt",
                        "ref.txt"
                    ],
                    token=HF_TOKEN
                )
                local_model_path = repo_dir
                os.makedirs(local_model_path, exist_ok=True)
                
                # ── DYNAMIC CONFIG HEALER (Fixes 1024-dim vs 2048-dim bias mismatch) ──
                # SFT checkpoint config is 1024-dim (from CustomVoice), but weights contain 2048-dim Base speaker encoder.
                # We dynamically copy the speaker_encoder_config block from official base cache to prevent size mismatch crashes.
                sft_config_path = os.path.join(local_model_path, "config.json")
                if os.path.exists(sft_config_path):
                    try:
                        with open(sft_config_path, "r", encoding="utf-8") as f_cfg:
                            sft_config = json.load(f_cfg)
                        
                        base_config_path = os.path.join(base_dir, "config.json")
                        if os.path.exists(base_config_path):
                            with open(base_config_path, "r", encoding="utf-8") as f_base:
                                base_config = json.load(f_base)
                            
                            sft_config["speaker_encoder_config"] = base_config["speaker_encoder_config"]
                            with open(sft_config_path, "w", encoding="utf-8") as f_out:
                                json.dump(sft_config, f_out, indent=2, ensure_ascii=False)
                            print("🔧 [Config Healer] Dynamic speaker_encoder_config alignment complete (patched to 2048-dim).", flush=True)
                    except Exception as e_heal:
                        print(f"⚠️ [Config Healer Warning] Failed to align config: {e_heal}", flush=True)
                
                tokenizer_files = ["tokenizer_config.json", "vocab.json", "merges.txt"]
                for f_name in tokenizer_files:
                    target_file_path = os.path.join(local_model_path, f_name)
                    if not os.path.exists(target_file_path):
                        src_file = os.path.join(base_dir, f_name)
                        if os.path.exists(src_file):
                            shutil.copy2(src_file, target_file_path)

                base_speech_tok_dir = os.path.join(base_dir, "speech_tokenizer")
                target_speech_tok_dir = os.path.join(local_model_path, "speech_tokenizer")
                if os.path.exists(base_speech_tok_dir) and not os.path.exists(target_speech_tok_dir):
                    try:
                        os.symlink(base_speech_tok_dir, target_speech_tok_dir)
                        print("🔗 [Symlink] Created speech_tokenizer link to base model.", flush=True)
                    except Exception as e:
                        print(f"⚠️ [Symlink Warning] Symlink failed: {e}. Falling back to copy...", flush=True)
                        try:
                            shutil.copytree(base_speech_tok_dir, target_speech_tok_dir)
                        except Exception as copy_err:
                            print(f"❌ [Copy Error] Failed to copy tokenizer: {copy_err}", flush=True)

                print(f"📁 Loading model locally from cache directory: '{local_model_path}'", flush=True)
                current_tts_model = Qwen3TTSModel.from_pretrained(
                    local_model_path,
                    **kwargs
                )
                
                # Keep model loading in base (ICL Cloning) mode if tts_model_type in config.json is base
                if hasattr(current_tts_model.model, "config") and getattr(current_tts_model.model.config, "tts_model_type", "base") == "base":
                    print("ℹ️ Full SFT Model loaded in Hybrid Base (ICL Cloning) mode.", flush=True)
                    current_tts_model.tts_model_type = "base"
                    current_tts_model.model.tts_model_type = "base"
                else:
                    print("ℹ️ Full SFT Model loaded in Standard CustomVoice (Index-based) mode.", flush=True)
                    current_tts_model.tts_model_type = "custom_voice"
                    if hasattr(current_tts_model, "model"):
                        current_tts_model.model.tts_model_type = "custom_voice"
                        if hasattr(current_tts_model, "model") and hasattr(current_tts_model.model.config, "talker_config"):
                            talker_cfg = current_tts_model.model.config.talker_config
                            if talker_cfg and hasattr(talker_cfg, "spk_id") and talker_cfg.spk_id:
                                current_tts_model.supported_speakers = list(talker_cfg.spk_id.keys())
        
        # ── EXPLICIT HARDWARE ACCELERATION CAST (Fixes 100% CPU lockup) ──
        if IS_GPU:
            print(f"⚡ [GPU] Casting wrapped models explicitly to hardware device '{DEVICE}'...", end="", flush=True)
            current_tts_model.model.to(DEVICE)
            if hasattr(current_tts_model, "device"):
                current_tts_model.device = DEVICE
            print(" Done.")
        else:
            current_tts_model.model.to("cpu")
            
        current_tts_model_path = local_model_path
        current_tts_type = target_model_type
        current_tts_subfolder = target_subfolder
        active_loading_subfolder = None
        return current_tts_model
    except Exception as e:
        print(f"❌ Error loading TTS model: {e}", flush=True)
        active_loading_subfolder = None
        return None

def voice_clone(text, reference_audio, ref_transcript, use_fast_mode, temperature=0.8, repetition_penalty=1.05, top_p=0.9, top_k=50, do_sample=True):
    if not text or not reference_audio: return None
    try:
        model = load_tts_model("base")
        if model is None: return None
        with torch.inference_mode():
            prompt = model.create_voice_clone_prompt(ref_audio=reference_audio, x_vector_only_mode=use_fast_mode or not ref_transcript, ref_text=None if use_fast_mode else ref_transcript)
            wavs, sr = model.generate_voice_clone(
                text=text,
                voice_clone_prompt=prompt,
                temperature=float(temperature),
                repetition_penalty=float(repetition_penalty),
                top_p=float(top_p),
                top_k=int(top_k),
                do_sample=bool(do_sample),
                max_new_tokens=512  # Forced safe cap to prevent infinite loop hangs
            )
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wavs[0], sr)
        return temp_file.name
    except Exception: return None

def custom_voice(text, voice_name, instruction, lora_scale=0.8, temperature=0.6, repetition_penalty=1.05, top_p=0.8, top_k=20, seed=-1, do_sample=True, language="auto"):
    global current_tts_model_path
    if not text: return None
    try:
        import contextlib
        
        orig_benchmark = torch.backends.cudnn.benchmark if IS_GPU else True
        if int(seed) >= 0:
            print(f"🎲 [Seed] Setting global random seed to {int(seed)} and enforcing determinism...", flush=True)
            torch.manual_seed(int(seed))
            if IS_GPU:
                torch.cuda.manual_seed_all(int(seed))
                torch.backends.cudnn.deterministic = True
                torch.backends.cudnn.benchmark = False

        standard_presets = ["Serena", "Vivian", "Ono_Anna", "Sohee", "Aiden", "Dylan", "Eric", "Ryan", "Uncle_Fu"]

        if voice_name in standard_presets:
            model = load_tts_model("custom")
            active_speaker = voice_name
            is_hybrid_clone = False
        else:
            if voice_name == "---":
                print("⚠️ Invalid voice selection: '---' is a divider.", flush=True)
                return None
            model = load_tts_model("finetuned", subfolder=voice_name)
            active_speaker = voice_name
            
            # Checks if the loaded SFT model runs in hybrid cloning (base) mode
            is_hybrid_clone = getattr(model.model, "tts_model_type", "base") == "base"

        if model is None: return None
        
        is_peft = hasattr(model.model, "peft_config")
        
        # Clean and map language ISO codes dynamically
        lang_mapping = {
            "en": "english",
            "zh": "chinese",
            "auto": "auto"
        }
        clean_lang = lang_mapping.get(str(language).lower().split("-")[0], str(language).lower().strip())
        if not clean_lang or clean_lang == "auto":
            clean_lang = "auto"

        with torch.inference_mode():
            if is_peft:
                from peft.helpers import rescale_adapter_scale
                scale_ctx = rescale_adapter_scale(model.model, float(lora_scale))
            else:
                scale_ctx = contextlib.nullcontext()
                
            with scale_ctx:
                warm_text = " " + str(text).strip()
                
                if is_hybrid_clone:
                    # ── SFT + In-Context Learning (ICL) Cloning Pathway ──
                    print("🎭 Hybrid SFT Mode: Synthesis running in SFT + In-Context Learning (ICL) Cloning mode...", flush=True)
                    
                    local_model_path = current_tts_model_path
                    
                    ref_names = ["ref_sample.wav", "ref.wav"]
                    ref_path = None
                    for r_name in ref_names:
                        cand = os.path.join(local_model_path, r_name)
                        if os.path.exists(cand):
                            ref_path = cand
                            break
                    
                    if not ref_path:
                        raise ValueError(f"Could not locate reference audio (ref.wav or ref_sample.wav) in SFT directory: {local_model_path}")
                    
                    ref_txt_names = ["ref_sample.txt", "ref.txt"]
                    ref_text = ""
                    for t_name in ref_txt_names:
                        cand = os.path.join(local_model_path, t_name)
                        if os.path.exists(cand):
                            with open(cand, "r", encoding="utf-8") as f_ref:
                                ref_text = f_ref.read().strip()
                            break
                    
                    # Fix: Dynamic SFT+ICL Style Injector (Pre-tokenization XML tagging)
                    if instruction and len(instruction.strip()) > 0:
                        warm_text = f"<instruct>{instruction.strip()}</instruct> " + warm_text
                        print(f"🎨 [Style Inject] Pre-processed SFT prompt: '{warm_text}'", flush=True)

                    print(f"🎤 [Hybrid ICL] Synthesizing voice clone with reference audio: '{ref_path}'", flush=True)
                    prompt = model.create_voice_clone_prompt(
                        ref_audio=ref_path,
                        x_vector_only_mode=False,
                        ref_text=ref_text if ref_text else None
                    )
                    
                    # Dynamic Text-Proportional Token Limiter based on character length
                    # Ensures short text terminates quickly while granting long text sufficient headroom
                    char_count = len(str(text))
                    dynamic_tokens = min(4096, max(250, int(round(char_count * 3.75))))
                    print(f"🔋 [Token Limiter] Characters: {char_count} | Dynamic safe ceiling: {dynamic_tokens} tokens", flush=True)
                    
                    wavs, sr = model.generate_voice_clone(
                        text=warm_text,
                        voice_clone_prompt=prompt,
                        temperature=float(temperature),
                        repetition_penalty=float(repetition_penalty),
                        top_p=float(top_p),
                        top_k=int(top_k),
                        do_sample=bool(do_sample),
                        max_new_tokens=dynamic_tokens  # Replaced static 512 with dynamic token ceiling
                    )
                else:
                    # Standard CustomVoice index-based SFT generation
                    if is_peft:
                        print(f"🎭 SFT Custom Voice: Synthesis running with LoRA scale {lora_scale}...", flush=True)
                    else:
                        print("🎭 SFT Custom Voice: Synthesis running in Full SFT mode (Unscaled)...", flush=True)
                    
                    # Dynamic Text-Proportional Token Limiter based on character length
                    char_count = len(str(text))
                    dynamic_tokens = min(4096, max(250, int(round(char_count * 3.75))))
                    print(f"🔋 [Token Limiter] Characters: {char_count} | Dynamic safe ceiling: {dynamic_tokens} tokens", flush=True)
                    
                    # Native Style Prompting: pass instruct argument cleanly to prevent reading aloud
                    wavs, sr = model.generate_custom_voice(
                        text=warm_text,
                        language=clean_lang,
                        speaker=active_speaker,
                        instruct=instruction if instruction else None,
                        temperature=float(temperature),
                        repetition_penalty=float(repetition_penalty),
                        top_p=float(top_p),
                        top_k=int(top_k),
                        do_sample=bool(do_sample),
                        max_new_tokens=dynamic_tokens  # Replaced static 512 with dynamic token ceiling
                    )

        if IS_GPU and int(seed) >= 0:
            torch.backends.cudnn.benchmark = orig_benchmark
            torch.backends.cudnn.deterministic = False
            
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wavs[0], sr)
        return temp_file.name
    except Exception as e:
        print(f"❌ Custom Voice Generation Error: {e}", flush=True)
        if IS_GPU and 'orig_benchmark' in locals() and int(seed) >= 0:
            torch.backends.cudnn.benchmark = orig_benchmark
            torch.backends.cudnn.deterministic = False
        return None

def voice_design(text, voice_description, temperature=0.8, repetition_penalty=1.05, top_p=0.9, top_k=50, do_sample=True):
    if not text or not voice_description: return None
    try:
        model = load_tts_model("design")
        if model is None: return None
        with torch.inference_mode():
            # Dynamic Text-Proportional Token Limiter based on character length
            char_count = len(str(text))
            dynamic_tokens = min(4096, max(250, int(round(char_count * 3.75))))
            print(f"🔋 [Token Limiter] Characters: {char_count} | Dynamic safe ceiling: {dynamic_tokens} tokens", flush=True)
            
            wavs, sr = model.generate_voice_design(
                text=text,
                instruct=voice_description,
                temperature=float(temperature),
                repetition_penalty=float(repetition_penalty),
                top_p=float(top_p),
                top_k=int(top_k),
                do_sample=bool(do_sample),
                max_new_tokens=dynamic_tokens  # Replaced static 512 with dynamic token ceiling
            )
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wavs[0], sr)
        return temp_file.name
    except Exception: return None

class ProductionAuditEngine:
    """
    Universal Aligner with Robust Manual Timing Correction.
    Finds sound onset via RMS scan to override AI origin-snapping.
    """
    def __init__(self):
        global current_audit_model
        self.model = None

    def _find_sound_onset(self, audio_path, threshold_db=-40):
        try:
            data, sr = sf.read(audio_path)
            if len(data.shape) > 1: data = np.mean(data, axis=1)
            energy = np.abs(data)
            peak = np.max(energy)
            if peak == 0: return 0.0
            limit = peak * (10**(threshold_db / 20))
            indices = np.where(energy > limit)[0]
            if len(indices) > 0: return indices[0] / sr
        except Exception: pass
        return 0.0

    def load(self):
        global current_audit_model
        if current_audit_model is not None: return current_audit_model
        
        if not KEEP_WARM:
            unload_tts_model()
            
        print("📥 Loading Stable-TS (large-v3) Aligner...")
        current_audit_model = stable_whisper.load_faster_whisper("large-v3", device=DEVICE.split(":")[0], compute_type=COMPUTE_WHISPER)
        return current_audit_model

    def transcribe(self, audio_path, language, prompt, beam_size, vad, temperature):
        if not audio_path: return {}, {}
        model = self.load()
        start_time = time.time()

        physical_onset = self._find_sound_onset(audio_path)

        active_lang = str(language).lower().strip()
        if active_lang == "auto" or not active_lang:
            print("🔍 Detecting language...", end="", flush=True)
            _, info = model.model.transcribe(audio_path, beam_size=1)
            active_lang = info.language
            print(f" Detected: {active_lang}")

        if prompt and len(prompt.strip()) > 10:
            print(f"🚀 Aligner: Forced Alignment started [Onset Detect: {physical_onset:.2f}s]")
            result = model.align(audio_path, prompt, language=active_lang)
        else:
            print(f"🚀 Aligner: Free Transcription started [Onset Detect: {physical_onset:.2f}s]")
            result = model.transcribe(
                audio_path, language=active_lang, initial_prompt=prompt,
                word_timestamps=True, beam_size=int(beam_size),
                vad=bool(vad), temperature=float(temperature),
                condition_on_previous_text=False
            )
            
        word_objs = result.all_words()
        shift_val = 0.0
        if word_objs:
            ai_start = word_objs[0].start
            if ai_start < physical_onset:
                diff = physical_onset - ai_start
                if diff > 0.05:
                    shift_val = diff
                    print(f"   > Correcting Origin Snap: Adding {shift_val:.2f}s manual offset")
        
        whisper_json = {
            "text": result.text.strip(),
            "segments": [{"start": round(s.start + shift_val, 2), "end": round(s.end + shift_val, 2), "text": s.text.strip()} for s in result.segments]
        }

        stable_json = {
            "text": result.text.strip(),
            "words": [],
            "audit": {
                "onset": round(physical_onset, 2),
                "applied_shift": round(shift_val, 2),
                "sec": round(time.time() - start_time, 2)
            }
        }
        for word in word_objs:
            stable_json["words"].append({
                "word": word.word.strip(),
                "start": round(word.start + shift_val, 2),
                "end": round(word.end + shift_val, 2),
                "confidence": round(getattr(word, 'probability', 1.0), 2)
            })

        print(f"✅ Finished in {stable_json['audit']['sec']}s")
        return whisper_json, stable_json

audit_engine = ProductionAuditEngine()

css = "\n.gradio-container { background-color: #f8fafc; }\n.footer { text-align: center; padding: 20px; color: #64748b; font-size: 0.9em; }\n.audit-header { background: #cbd5e1; padding: 8px 12px; border-radius: 6px; border-left: 5px solid #4338ca; margin-bottom: 8px; font-weight: bold; color: #1e1b4b; font-size: 0.9em; }\n"

try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    refs = api.list_repo_refs(repo_id=HF_REPO_ID)
    custom_subfolders = sorted([b.name for b in refs.branches if b.name != "main"])
    print(f"📦 [Scan] Discovered {len(custom_subfolders)} custom voices in repo: {custom_subfolders}", flush=True)
except Exception as e:
    print(f"⚠️ [Scan Warning] Failed to scan HF repo directories: {e}", flush=True)
    custom_subfolders = []

standard_presets = ["Serena", "Vivian", "Ono_Anna", "Sohee", "Aiden", "Dylan", "Eric", "Ryan", "Uncle_Fu"]

if custom_subfolders:
    voice_dropdown_choices = custom_subfolders + ["---"] + standard_presets
    default_voice = custom_subfolders[0]
else:
    voice_dropdown_choices = standard_presets
    default_voice = "Ryan"

with gr.Blocks(title="Watch The Book - Unified", theme=gr.themes.Soft(), css=css) as demo:
    gr.HTML(f"""
        <div style='text-align: center; padding: 25px; background: #1e1b4b; border-radius: 12px; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
            <h1 style='margin: 0; color: white; font-size: 2.2em;'>🎙️ Watch The Book: Unified Production</h1>
            <p style='font-size: 1.1em; color: #94a3b8;'>Qwen3-TTS Generation & Universal High-Precision Audit</p>
            <p style='background: rgba(255,255,255,0.1); display: inline-block; padding: 5px 15px; border-radius: 5px; color: #fbbf24; font-weight: bold; margin-top: 10px;'>
                Hardware: {HW_STATUS}</p>
        </div>
    """)

    with gr.Tab("🎤 Voice Cloning"):
        with gr.Row():
            with gr.Column():
                c_text = gr.Textbox(label="Text to Synthesize", lines=4)
                c_audio = gr.Audio(label="Reference Audio", type="filepath")
                c_trans = gr.Textbox(label="Transcript (Optional)")
                c_fast = gr.Checkbox(label="Fast Mode", value=True)
                c_btn = gr.Button("🎵 Generate Speech", variant="primary", size="lg")
                with gr.Accordion("Advanced Parameters", open=False):
                    c_temp = gr.Slider(label="Temperature", minimum=0.1, maximum=1.5, value=0.8, step=0.05)
                    c_rep = gr.Slider(label="Repetition Penalty", minimum=1.0, maximum=3.0, value=1.05, step=0.05)
                    c_top_p = gr.Slider(label="Top P", minimum=0.1, maximum=1.0, value=0.9, step=0.05)
                    c_top_k = gr.Slider(label="Top K", minimum=1, maximum=100, value=50, step=1)
                    c_sample = gr.Checkbox(label="Do Sample", value=True)
            with gr.Column():
                c_out = gr.Audio(label="Generated Output")
        c_btn.click(voice_clone, [c_text, c_audio, c_trans, c_fast, c_temp, c_rep, c_top_p, c_top_k, c_sample], c_out, api_name="voice_clone")

    with gr.Tab("🎭 Custom Voice"):
        with gr.Row():
            with gr.Column():
                cust_text = gr.Textbox(label="Text to Synthesize", lines=4)
                cust_name = gr.Dropdown(choices=voice_dropdown_choices, label="Character", value=default_voice, allow_custom_value=True)
                cust_inst = gr.Textbox(label="Style Instruction")
                cust_lang = gr.Dropdown(choices=["auto", "english", "chinese", "french", "german", "italian", "japanese", "korean", "portuguese", "russian", "spanish"], label="Language", value="auto")
                cust_btn = gr.Button("🎵 Generate Speech", variant="primary", size="lg")
                with gr.Accordion("Advanced Parameters", open=False):
                    cust_scale = gr.Slider(label="LoRA Adapter Scale (0.3=Standard, 0.8=Strong)", minimum=0.1, maximum=1.0, value=0.8, step=0.05)
                    cust_seed = gr.Number(label="Global Seed (-1 for Random)", value=-1, precision=0)
                    cust_temp = gr.Slider(label="Temperature", minimum=0.1, maximum=1.5, value=0.6, step=0.05)
                    cust_rep = gr.Slider(label="Repetition Penalty", minimum=1.0, maximum=3.0, value=1.05, step=0.05)
                    cust_top_p = gr.Slider(label="Top P", minimum=0.1, maximum=1.0, value=0.8, step=0.05)
                    cust_top_k = gr.Slider(label="Top K", minimum=1, maximum=100, value=20, step=1)
                    cust_sample = gr.Checkbox(label="Do Sample", value=True)
            with gr.Column():
                cust_out = gr.Audio(label="Generated Output")
        cust_btn.click(custom_voice, [cust_text, cust_name, cust_inst, cust_scale, cust_temp, cust_rep, cust_top_p, cust_top_k, cust_seed, cust_sample, cust_lang], cust_out, api_name="custom_voice")

    with gr.Tab("🎨 Voice Design"):
        with gr.Row():
            with gr.Column():
                d_text = gr.Textbox(label="Text to Synthesize", lines=4)
                d_desc = gr.Textbox(label="Voice Description")
                d_btn = gr.Button("🎨 Generate Speech", variant="primary", size="lg")
                with gr.Accordion("Advanced Parameters", open=False):
                    d_temp = gr.Slider(label="Temperature", minimum=0.1, maximum=1.5, value=0.8, step=0.05)
                    d_rep = gr.Slider(label="Repetition Penalty", minimum=1.0, maximum=3.0, value=1.05, step=0.05)
                    d_top_p = gr.Slider(label="Top P", minimum=0.1, maximum=1.0, value=0.9, step=0.05)
                    d_top_k = gr.Slider(label="Top K", minimum=1, maximum=100, value=50, step=1)
                    d_sample = gr.Checkbox(label="Do Sample", value=True)
            with gr.Column():
                d_out = gr.Audio(label="Generated Output")
        d_btn.click(voice_design, [d_text, d_desc, d_temp, d_rep, d_top_p, d_top_k, d_sample], d_out, api_name="voice_design")

    with gr.Tab("📝 Quality Audit"):
        with gr.Row():
            with gr.Column(scale=1):
                a_audio = gr.Audio(label="Audio Source", type="filepath")
                a_lang = gr.Textbox(label="Language ISO (or 'auto')", value="auto")
                a_prompt = gr.Textbox(label="Manuscript Hint (Triggers Forced Alignment)", lines=4)
                a_btn = gr.Button("📡 Run Precision Audit", variant="primary", size="lg")
                with gr.Accordion("Parameters", open=False):
                    a_beam = gr.Number(label="Beam Size", value=5)
                    a_vad = gr.Checkbox(label="Use VAD", value=False)
                    a_temp = gr.Number(label="Temperature", value=0.0)
            with gr.Column(scale=2):
                gr.HTML("<div class='audit-header'>Overview: Standard Segments (Small Window)</div>")
                out_small = gr.JSON(label="Segments Result", height=200)
                gr.HTML("<div class='audit-header'>Precision: Aligned Word Timings (Big Window)</div>")
                out_big = gr.JSON(label="Aligned JSON Result", height=600)

        a_btn.click(audit_engine.transcribe, [a_audio, a_lang, a_prompt, a_beam, a_vad, a_temp], [out_small, out_big], api_name="transcribe")

    gr.HTML("<div class='footer'>🎓 Production Architecture for Youtube Watch The Book — Precision Audio Logic</div>")

demo.launch(share=True, debug=True, max_threads=10)
